In [9]:
from infrastructure.llm_client import create_llm_client
from lcb_runner.benchmarks import code_generation

dataset = code_generation.load_code_generation_dataset(release_version="release_v5", start_date="2024-01-01", difficulty=code_generation.Difficulty.HARD)
sample_problem = dataset[0]
llm_client = create_llm_client(provider="openai", model="gpt-5")

2025-10-28 12:26:02.085 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:143 - Using 'test' split of the dataset.
2025-10-28 12:26:25.455 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:155 - Filtered problems by difficulty: hard
2025-10-28 12:26:25.490 | INFO     | lcb_runner.benchmarks.code_generation:load_code_generation_dataset:165 - Loaded 194 problems
2025-10-28 12:26:25.593 | INFO     | common.llm_client:create_llm_client:371 - Creating LLM client: provider=openai, model=gpt-5
2025-10-28 12:26:25.594 | DEBUG    | common.llm_client:__init__:43 - Initialized LLM client: model=gpt-5, max_tokens=1000000, limit_enabled=True
2025-10-28 12:26:25.600 | DEBUG    | common.llm_client:__init__:247 - Initialized OpenAIClient with model: gpt-5, reasoning_effort: minimal
2025-10-28 12:26:25.600 | INFO     | common.llm_client:create_llm_client:403 - Successfully created OpenAIClient


# Sample Problem Analysis

In [10]:
print("Problem: ")
print(sample_problem.question_content)

print("Public Test Cases: ")
for test_case in sample_problem.public_test_cases:
    print(test_case)

print("Starter Code: ")
print(sample_problem.starter_code)


Problem: 
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to change at most one index in s to another lowercase English letter.
Return an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.
 
Example 1:

Input: s = "accca", k = 2
Output: 3
Explanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to 'b'.
s becomes "acbca".
The operations can now be performed as follows until s becomes empty:
- Choose the longest prefix containing at most 2 distinct characters, "acbca".
- Delete the prefix, and s becomes "bca". The number of partitions is no

# Generate Initial Populations

In [11]:
import importlib
from coevolution.population import TestPopulation, CodePopulation
from infrastructure.code_preprocessing.analyzers import extract_test_methods_code
from coevolution.bayesian import initialize_prior_beliefs
from coevolution.config import CoevolutionConfig
import coevolution.operators as operators

importlib.reload(operators)

llm_model = "gpt-5"
config = CoevolutionConfig(
        # Population sizes
        initial_code_population_size=2,
        initial_test_population_size=5,
        max_code_population_size=15,  # population grows over time to 15
        # Bayesian parameters
        initial_code_prior=0.5,
        initial_test_prior=0.5,
        alpha=0.01,
        beta=0.2,
        gamma=0.2,
        learning_rate=0.1,
        use_intermediate_updates=False,
        # Evolution parameters
        num_generations=3,  # Small number for quick testing
        selection_strategy="roulette_wheel",
        # Code genetic operators
        code_crossover_rate=0.2,
        code_mutation_rate=0.1,
        code_edit_rate=0.8,
        code_elite_proportion=0.5,
        code_offspring_proportion=0.5,
        # Test genetic operators
        test_crossover_rate=0.5,
        test_mutation_rate=0.3,
        test_edit_rate=0.5,
        # LLM configuration
        llm_model=llm_model,
)


code_operator = operators.CodeOperator(llm_client=llm_client, problem=sample_problem)
test_operator = operators.TestOperator(llm_client=llm_client, problem=sample_problem)

# Create initial populations
code_individuals = code_operator.create_initial_population(config.initial_code_population_size)
code_probs = initialize_prior_beliefs(
    config.initial_code_population_size, config.initial_code_prior
)

test_class_block = test_operator.create_initial_population(config.initial_test_population_size)
test_individuals = extract_test_methods_code(test_class_block)

# Initialize prior probabilities
test_probs = initialize_prior_beliefs(
    config.initial_test_population_size, config.initial_test_prior
)


code_population = CodePopulation(
    individuals=code_individuals,
    probabilities=code_probs,
    generation=0,
)

# Create population
test_population = TestPopulation(
    individuals=test_individuals,
    probabilities=test_probs,
    generation=0,
    test_class_block=test_class_block,
)


print("Code Population Individuals:")
for (code, prob) in code_population:
    print(f"Probability: {prob:.4f}\nCode:\n{code}\n")

print("\nTest Population Individuals:")
for (test, prob) in test_population:
    print(f"Probability: {prob:.4f}\nTest Method:\n{test}\n")

2025-10-28 12:26:25.614 | INFO     | common.coevolution.operators:create_initial_population:630 - CodeOperator: Creating initial population of size 2
2025-10-28 12:26:25.615 | DEBUG    | common.coevolution.operators:_generate_with_retry:445 - Sending prompt to LLM (requesting 2 items)
2025-10-28 12:26:25.615 | DEBUG    | common.llm_client:generate:256 - OpenAIClient generating with model: gpt-5, reasoning_effort: minimal
2025-10-28 12:27:10.137 | DEBUG    | common.llm_client:generate:276 - Generated 10847 characters
2025-10-28 12:27:10.137 | DEBUG    | common.coevolution.operators:_generate_with_retry:454 - Received response from LLM
2025-10-28 12:27:10.138 | DEBUG    | common.coevolution.operators:_generate_with_retry:464 - Extracted 2 code blocks from response
2025-10-28 12:27:10.140 | INFO     | common.coevolution.operators:create_initial_population:648 - CodeOperator: Successfully generated 2 solutions for initial population
2025-10-28 12:27:10.141 | DEBUG    | common.coevolution.b

Code Population Individuals:
Probability: 0.5000
Code:
class Solution:
    def maxPartitionsAfterOperations(self, s: str, k: int) -> int:
        # Solution 1: Greedy simulation with one optimal change
        #
        # Idea:
        # - Without any change, the greedy algorithm is:
        #   scan from left to right, keep the longest prefix with <= k distinct chars,
        #   when adding next char would exceed k distinct, cut a partition and start new.
        # - With one change allowed, the only place a change can help is when a "violation"
        #   would occur (i.e., we’d need to cut). If we can change the current char to
        #   any already-seen character within the current window, we avoid cutting here,
        #   possibly resulting in fewer cuts later and thus more total partitions overall across the string.
        #   Actually, more partitions means more cuts; but by avoiding a forced cut early, we can extend the current
        #   prefix and later force more cuts

# Execute the tests

In [12]:
import common.sandbox as sandbox_module
import coevolution.evaluation as evaluation
importlib.reload(sandbox_module)
importlib.reload(evaluation)

sandbox = sandbox_module.create_safe_test_environment()
exec_results = evaluation.execute_code_against_tests(code_population, test_population, sandbox)
observation_matrix = evaluation.generate_observation_matrix(exec_results, len(code_population), len(test_population))

print("Observation Matrix:")
print(observation_matrix)

2025-10-28 12:27:21.101 | INFO     | common.coevolution.evaluation:execute_code_against_tests:60 - Executing code against tests: 2 code × 5 tests = 10 evaluations
2025-10-28 12:27:21.102 | DEBUG    | common.coevolution.evaluation:execute_code_against_tests:67 - Evaluating code snippet 1/2
2025-10-28 12:27:21.104 | DEBUG    | common.sandbox:execute_test_script:817 - SafeCodeSandbox: executing test script (len=2193)
2025-10-28 12:27:21.105 | DEBUG    | common.sandbox:execute_code:736 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp6510qp4v.py capture_output=True code_len=2204
2025-10-28 12:27:21.208 | DEBUG    | common.sandbox:execute_code:752 - Execution finished: returncode=1
2025-10-28 12:27:21.209 | DEBUG    | common.sandbox:analyze_unittest_output:138 - Analyzing unittest output: success=False return_code=1 elapsed=0.104s
2025-10-28 12:27:21.209 | DEBUG    | common.sandbox:analyze_unittest_output:163 - Test run had failures/errors: Tests co

Observation Matrix:
[[1 1 0 1 0]
 [1 1 0 1 0]]


# Update Beliefs

In [13]:
from coevolution.bayesian import update_population_beliefs


# Pass rates
code_pass_rates = evaluation.compute_code_pass_rates(observation_matrix)
test_pass_rates = evaluation.compute_test_pass_rates(observation_matrix)

print("Code Population Pass Rates:")
print(code_pass_rates)

print("Test Population Pass Rates:")
print(test_pass_rates)

discriminations = evaluation.compute_test_discriminations(observation_matrix)
print("Test Population Discriminations:")
print(discriminations)

# Update beliefs
posterior_code_probs, posterior_test_probs = update_population_beliefs(code_population.probabilities, test_population.probabilities, observation_matrix, config)

code_population.update_probabilities(posterior_code_probs)
test_population.update_probabilities(posterior_test_probs)

print("\nUpdated Code Population Probabilities:")
print(code_population.probabilities)

print("\nUpdated Test Population Probabilities:")
print(test_population.probabilities)

2025-10-28 12:27:21.269 | DEBUG    | common.coevolution.evaluation:compute_code_pass_rates:148 - Computing code pass rates for matrix with shape (2, 5)
2025-10-28 12:27:21.269 | DEBUG    | common.coevolution.evaluation:compute_code_pass_rates:159 - Computed code pass rates, returning array with shape (2,)
2025-10-28 12:27:21.270 | DEBUG    | common.coevolution.evaluation:compute_test_pass_rates:176 - Computing test pass rates for matrix with shape (2, 5)
2025-10-28 12:27:21.270 | DEBUG    | common.coevolution.evaluation:compute_test_pass_rates:187 - Computed test pass rates, returning array with shape (5,)
2025-10-28 12:27:21.271 | DEBUG    | common.coevolution.evaluation:compute_test_discriminations:209 - Computing test discriminations for matrix with shape (2, 5)
2025-10-28 12:27:21.271 | DEBUG    | common.coevolution.evaluation:compute_test_pass_rates:176 - Computing test pass rates for matrix with shape (2, 5)
2025-10-28 12:27:21.271 | DEBUG    | common.coevolution.evaluation:compu

Code Population Pass Rates:
[0.6 0.6]
Test Population Pass Rates:
[1. 1. 0. 1. 0.]
Test Population Discriminations:
[4.13049503e-11 4.13049503e-11 4.13058003e-11 4.13049503e-11
 4.13058003e-11]

Updated Code Population Probabilities:
[0.54534049 0.54534049]

Updated Test Population Probabilities:
[0.58627655 0.58627655 0.45981888 0.58627655 0.45981888]


# Elites

In [14]:
num_code_elites = int(config.code_elite_proportion * len(code_population))
code_elites = code_population.get_top_k_individuals(num_code_elites)

print("\nCode Population Elites:")
for elite in code_elites:
    print(elite)

test_population.set_discriminations(discriminations)
test_pareto_front = test_population.get_pareto_front()
print("\nTest Population Pareto Front Individuals:")
for individual in test_pareto_front:
    print(individual)

2025-10-28 12:27:21.277 | DEBUG    | common.coevolution.population:set_discriminations:545 - Setting new discriminations for 5 individuals
2025-10-28 12:27:21.277 | DEBUG    | common.coevolution.population:set_discriminations:584 - Updated discriminations for generation 0: avg 0.0000 → 0.0000 (Δ+0.0000)
2025-10-28 12:27:21.277 | DEBUG    | common.coevolution.population:get_pareto_front:602 - Calculating Pareto front...
2025-10-28 12:27:21.278 | DEBUG    | common.coevolution.selection:pareto_front:174 - Pareto front: selected 2 points, prob range=[0.4598, 0.5863], disc range=[0.0000, 0.0000]
2025-10-28 12:27:21.278 | DEBUG    | common.coevolution.population:get_pareto_front:626 - Found 2 initial Pareto-optimal individuals
2025-10-28 12:27:21.278 | DEBUG    | common.coevolution.population:get_pareto_front:630 - Filtering front by average probability > 0.5357
2025-10-28 12:27:21.278 | DEBUG    | common.coevolution.population:get_pareto_front:635 - Filtered front from 2 to 1 individuals
20


Code Population Elites:
('class Solution:\n    def maxPartitionsAfterOperations(self, s: str, k: int) -> int:\n        # Solution 2: DP with state (position, used_change, current distinct mask size)\n        #\n        # Observations:\n        # - Each partition is the longest prefix with <= k distinct chars.\n        # - The choice at each position i depends only on the current partition’s set of distinct letters\n        #   and whether we\'ve used the change already.\n        # - When a new char would exceed k distinct, we must either:\n        #   - cut a partition here (and start a new one with this char), or\n        #   - if change not used, change this char to one already in the set and keep extending.\n        #\n        # We can model the process iteratively creating partitions:\n        # For each partition, we greedily extend to its maximal length given the ability to use at most one change\n        # globally. But this greedy choice (whether to spend the change) can branc

# Public Test Cases

In [15]:
import importlib
import common.code_preprocessing as cp
importlib.reload(cp.builders)
importlib.reload(evaluation)
importlib.reload(sandbox_module)


sandbox = sandbox_module.create_safe_test_environment()
public_test_block = cp.builders.build_unittest_block_for_lcb_problem_from_given_tests(sample_problem.public_test_cases, sample_problem.starter_code)
print("Generated Test Block:\n")
print(public_test_block)


public_test_individuals = extract_test_methods_code(public_test_block)

public_test_probs = initialize_prior_beliefs(
    len(public_test_individuals), 0.99
)

public_test_population = TestPopulation(
    individuals=public_test_individuals,
    probabilities=public_test_probs,
    generation=0,
    test_class_block=public_test_block,
)


public_exec_results = evaluation.execute_code_against_tests(code_population, public_test_population, sandbox)
public_observation_matrix = evaluation.generate_observation_matrix(public_exec_results, len(code_population), len(public_test_population))

print("Public Test Execution Results:")
print(public_exec_results)

print("Public Test Observation Matrix:")
print(public_observation_matrix)

Generated Test Block:


2025-10-28 12:27:21.286 | DEBUG    | common.coevolution.bayesian:initialize_prior_beliefs:225 - Initializing 3 prior beliefs with probability 0.9900
2025-10-28 12:27:21.286 | DEBUG    | common.coevolution.population:__init__:70 - Initialized TestPopulation: size=3, generation=0, avg_prob=0.9900
2025-10-28 12:27:21.286 | INFO     | common.coevolution.evaluation:execute_code_against_tests:60 - Executing code against tests: 2 code × 3 tests = 6 evaluations
2025-10-28 12:27:21.286 | DEBUG    | common.coevolution.evaluation:execute_code_against_tests:67 - Evaluating code snippet 1/2
2025-10-28 12:27:21.287 | DEBUG    | common.sandbox:execute_test_script:817 - SafeCodeSandbox: executing test script (len=2242)
2025-10-28 12:27:21.288 | DEBUG    | common.sandbox:execute_code:736 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp4xv4bxqs.py capture_output=True code_len=2242
2025-10-28 12:27:21.347 | DEBUG    | common.sandbox:execute_code:752 - Execution 


import unittest
from typing import * # Import common types for function signatures

class TestSolution(unittest.TestCase):
    def setUp(self):
        self.solution = Solution()

    def test_case_1(self):
        # Original Input: '"accca"\n2'
        input_lines = ['"accca"', '2']
        args = [eval(line) for line in input_lines]
        expected_output = eval('3')
        self.assertEqual(self.solution.maxPartitionsAfterOperations(*args), expected_output)

    def test_case_2(self):
        # Original Input: '"aabaab"\n3'
        input_lines = ['"aabaab"', '3']
        args = [eval(line) for line in input_lines]
        expected_output = eval('1')
        self.assertEqual(self.solution.maxPartitionsAfterOperations(*args), expected_output)

    def test_case_3(self):
        # Original Input: '"xxyz"\n1'
        input_lines = ['"xxyz"', '1']
        args = [eval(line) for line in input_lines]
        expected_output = eval('4')
        self.assertEqual(self.solution.maxPartitions

# Private Test Cases

In [16]:
private_test_block = cp.builders.build_unittest_block_for_lcb_problem_from_given_tests(sample_problem.private_test_cases, sample_problem.starter_code)

private_test_individuals = extract_test_methods_code(private_test_block)

private_test_probs = initialize_prior_beliefs(
    len(private_test_individuals), 0.99
)

private_test_population = TestPopulation(
    individuals=private_test_individuals,
    probabilities=private_test_probs,
    generation=0,
    test_class_block=private_test_block,
)


private_exec_results = evaluation.execute_code_against_tests(code_population, private_test_population, sandbox)
private_observation_matrix = evaluation.generate_observation_matrix(private_exec_results, len(code_population), len(private_test_population))

print("Private Test Observation Matrix:")
print(private_observation_matrix)

2025-10-28 12:27:21.412 | DEBUG    | common.coevolution.bayesian:initialize_prior_beliefs:225 - Initializing 12 prior beliefs with probability 0.9900
2025-10-28 12:27:21.413 | DEBUG    | common.coevolution.population:__init__:70 - Initialized TestPopulation: size=12, generation=0, avg_prob=0.9900
2025-10-28 12:27:21.413 | INFO     | common.coevolution.evaluation:execute_code_against_tests:60 - Executing code against tests: 2 code × 12 tests = 24 evaluations
2025-10-28 12:27:21.413 | DEBUG    | common.coevolution.evaluation:execute_code_against_tests:67 - Evaluating code snippet 1/2
2025-10-28 12:27:21.416 | DEBUG    | common.sandbox:execute_test_script:817 - SafeCodeSandbox: executing test script (len=34497)
2025-10-28 12:27:21.417 | DEBUG    | common.sandbox:execute_code:736 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpngt_q1q1.py capture_output=True code_len=34497
2025-10-28 12:27:21.477 | DEBUG    | common.sandbox:execute_code:752 - Exec

Private Test Observation Matrix:
[[1 0 0 0 1 1 0 1 1 1 0 1]
 [1 0 0 0 1 1 0 1 1 1 0 1]]
